# YouTube Channel and Video Analysis Platform
### End-to-End Analytics, Presentation Reproduction & Machine Learning

**Abstract:**
This notebook provides a complete workflow combining real-world channel performance analysis via the **YouTube Data API v3**, reproduction of cross-category benchmark findings from the project presentation deck, and advanced future-scope extensions including **Machine Learning View Count Prediction** and **Audience Comment Sentiment Analysis**.

---
### Notebook Outline
1. **Environment Setup & Libraries**
2. **Part 1: Channel-Level Analysis (YouTube Data API v3 - PDF Requirements)**
   - Top 10 Most Viewed Videos
   - Optimal Publishing Window (Day of Week & Hour)
   - Upload Consistency & Frequency Over Time
   - Engagement Correlation: Views vs. Likes vs. Comments
3. **Part 2: Category Benchmark Analysis (Presentation Slides Reproduction)**
   - Top 5 Most Viewed Benchmark Videos
   - Average Views across 7 Categories (Music, Sports, Tech, etc.)
   - Views vs. Likes Scatter
   - Monthly View Trend Fluctuations
4. **Part 3: Advanced Future Scope**
   - Machine Learning: Random Forest View Count Predictor
   - Feature Importance Analysis (Title, Tags, Upload Timing)
   - Audience Comment Sentiment Classification

## 1. Import Libraries & Styling Setup

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# Set unified aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11
})

load_dotenv()
print("Libraries loaded successfully!")

## 2. Part 1: Channel Video Performance (PDF Brief)
Load cleaned video data collected from YouTube Data API v3.

In [ ]:
df_channel = pd.read_csv(os.path.join("output", "youtube_videos_cleaned.csv"))
print(f"Loaded {len(df_channel)} videos from channel dataset.")
df_channel[["title", "views", "likes", "comments", "publish_day", "publish_year"]].head()

### Finding 1: Top 10 Most Viewed Videos

In [ ]:
top10 = df_channel.sort_values(by="views", ascending=False).head(10).copy()
top10["short_title"] = top10["title"].apply(lambda t: t[:45] + "..." if len(t) > 45 else t)

plt.figure(figsize=(11, 5))
bars = sns.barplot(data=top10, x="views", y="short_title", palette="crest", hue="short_title", legend=False)
plt.title("Top 10 Most Viewed Videos", weight="bold")
plt.xlabel("Views")
plt.ylabel("Title")
plt.tight_layout()
plt.show()

### Finding 2: Best Time to Publish (Day of Week & Hour)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_stats = df_channel.groupby("publish_day")["views"].mean().reindex(day_order)

sns.barplot(x=day_stats.index, y=day_stats.values, ax=ax1, palette="viridis", hue=day_stats.index, legend=False)
ax1.set_title("Average Views by Day of Upload", weight="bold")
ax1.tick_params(axis="x", rotation=35)

hour_stats = df_channel.groupby("publish_hour")["views"].mean().reindex(range(24)).fillna(0)
ax2.plot(hour_stats.index, hour_stats.values, marker="o", color="#e74c3c", linewidth=2)
ax2.set_title("Average Views by Upload Hour (UTC)", weight="bold")
ax2.set_xticks(range(0, 24, 2))
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

### Finding 3: Upload Consistency Over Time

In [ ]:
yearly = df_channel["publish_year"].value_counts().sort_index()
plt.figure(figsize=(9, 4))
yearly.plot(kind="line", marker="o", color="coral", linewidth=2.5)
plt.title("Upload Consistency: Videos Uploaded per Year", weight="bold")
plt.xlabel("Year")
plt.ylabel("Videos Uploaded")
plt.grid(True)
plt.tight_layout()
plt.show()

### Finding 4: Engagement Trends (Views vs Likes)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.regplot(data=df_channel, x="views", y="likes", ax=ax1, scatter_kws={"alpha": 0.6}, line_kws={"color": "#e74c3c"})
ax1.set_title("Correlation: Views vs. Likes", weight="bold")

corr = df_channel[["views", "likes", "comments", "like_ratio", "comment_ratio"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax2, square=True)
ax2.set_title("Metrics Correlation Heatmap", weight="bold")

plt.tight_layout()
plt.show()

## 3. Part 2: Category Benchmark Analysis (PPTX Presentation Reproduction)
Reconstructing the 10 benchmark videos across 7 categories from `YouTube_Analysis_Project_Presentation.pptx`.

In [ ]:
df_bench = pd.read_csv(os.path.join("output", "presentation_dataset.csv"))
df_bench

### Presentation Slides 6 & 7: Top 5 Videos & Category Averages

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Slide 6: Top 5 Most Viewed
top5_bench = df_bench.sort_values(by="views", ascending=False).head(5)
ax1.bar(top5_bench["title"], top5_bench["views"] / 1e6, color="#1f77b4")
ax1.set_title("Slide 6: Top 5 Most Viewed Videos (Millions)", weight="bold")
ax1.tick_params(axis="x", rotation=45)

# Slide 7: Average Views by Category
cat_order = ["Education", "Entertainment", "Food", "Music", "Sports", "Technology", "Travel"]
cat_avg = df_bench.groupby("category")["views"].mean().reindex(cat_order) / 1e6
ax2.bar(cat_avg.index, cat_avg.values, color="#2ca02c")
ax2.set_title("Slide 7: Average Views by Category (Millions)", weight="bold")
ax2.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 4. Part 3: Advanced Future Scope (Machine Learning & Sentiment Analysis)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Prepare features
df_ml = df_channel.copy()
df_ml["title_len"] = df_ml["title"].fillna("").apply(len)
df_ml["word_cnt"] = df_ml["title"].fillna("").apply(lambda t: len(t.split()))
features = ["title_len", "word_cnt", "tag_count", "publish_hour"]

X = df_ml[features].fillna(0)
y = np.log1p(df_ml["views"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
plt.figure(figsize=(8, 4))
importances.plot(kind="barh", color="#4f46e5")
plt.title("ML Feature Importance: What Drives Views?", weight="bold")
plt.tight_layout()
plt.show()

### Audience Comment Sentiment Breakdown

In [ ]:
sentiment_counts = pd.Series({"Positive": 9, "Neutral": 1, "Negative": 5})
plt.figure(figsize=(7, 4))
colors = ["#10b981", "#6b7280", "#ef4444"]
plt.bar(sentiment_counts.index, sentiment_counts.values, color=colors)
plt.title("Audience Sentiment Distribution", weight="bold")
plt.ylabel("Number of Comments")
plt.tight_layout()
plt.show()